In [10]:
import threading

#global variable on the heap
z = [ 0, 0 ]  

def zplus(x,y,idx):
    z[idx] = x+y

t1 = threading.Thread(target=zplus, args=(1,2,0))
t2 = threading.Thread(target=zplus, args=(2,3,1))
t1.start(); t2.start()
t1.join(); t2.join()

print (z)

[3, 5]


In [9]:
import multiprocessing as mp

#global variable on the heap
z = [ 0, 0 ]  

def zplus(x,y,idx):
    z[idx] = x+y
     
ctx = mp.get_context('fork')
ph = ctx.Process(target=zplus, args=(1,2,0))
pw = ctx.Process(target=zplus, args=(1,2,1))
ph.start(); pw.start()
ph.join(); pw.join()

print (z)

[0, 0]


In [12]:
import multiprocessing as mp
import threading

def echo_str(s,q):
    q.put(s)
    
hw = ['Hello', 'World']
ctx = mp.get_context('fork')
q = ctx.Queue()
ph = ctx.Process(target=echo_str, args=(hw[0],q))
pw = ctx.Process(target=echo_str, args=(hw[1],q))
ph.start(); pw.start()
print(q.get(), q.get(), '!')
ph.join(); pw.join()

Hello World !


In [13]:

# Adapted from:
# https://docs.python.org/3/library/multiprocessing.html#pipes-and-queues

import multiprocessing as mp
import threading
import time
import random

def echo_str(s,q):
    time.sleep(random.uniform(0,1))
    q.put(s)
    
def run_test():
    hw = ['Hello', 'World']
    ctx = mp.get_context('fork')
    q = ctx.Queue()
    ph = ctx.Process(target=echo_str, args=(hw[0],q))
    pw = ctx.Process(target=echo_str, args=(hw[1],q))
    ph.start(); pw.start()
    print(q.get(), q.get(), '!')
    ph.join(); pw.join()
    
for i in range(10):
    run_test()


Hello World !
World Hello !
World Hello !
World Hello !
World Hello !
Hello World !
Hello World !
World Hello !
World Hello !
World Hello !


In [31]:
import numpy as np
import random

def bitcount(num):
    return bin(num).count('1')
def bitcount_mt(buf, q):
    q.put(sum(map(bitcount,buf)))
    
np.random.seed(1)
buf = np.random.randint(0,1E9,int(1E6))

start_time = time.time()
sum_1s = sum(map(bitcount,buf))
end_time = time.time()
print("bitcount: %f seconds (single threaded)" 
      % (end_time - start_time))

start_time = time.time()
q = queue.Queue()
half_sz = int(len(buf)/2)
t1 = threading.Thread(target=bitcount_mt, args=(buf[:half_sz], q))
t2 = threading.Thread(target=bitcount_mt, args=(buf[half_sz:], q))
t1.start(); t2.start()
t1.join(); t2.join()
sum_1s = q.get() + q.get()
end_time = time.time()
print("bitcount: %f seconds (two threads)" % 
      (end_time - start_time))    


start_time = time.time()
ctx = mp.get_context('fork')
q = ctx.Queue()
p1 = ctx.Process(target=bitcount_mt, args=(buf[:half_sz], q))
p2 = ctx.Process(target=bitcount_mt, args=(buf[half_sz:], q))
p1.start(); p2.start()
p1.join(); p2.join()
sum_1s = q.get() + q.get()
end_time = time.time()
print("bitcount: %f seconds (two processes)" % 
      (end_time - start_time)) 

bitcount: 0.775873 seconds (single threaded)
bitcount: 0.705969 seconds (two threads)
bitcount: 0.459815 seconds (two processes)


In [19]:
buf = np.random.randint(0,1000,int(5e8))


In [21]:
start_time = time.time()
buf = buf+1
end_time = time.time()
print("buf+=1: %f seconds" % 
      (end_time - start_time)) 

start_time = time.time()
buf = buf+1
end_time = time.time()
print("buf+=1: %f seconds" % 
      (end_time - start_time)) 

buf+=1: 10.572802 seconds
buf+=1: 10.539239 seconds


In [25]:
import threading

def buf_plus(pbuf, start, end):
    buf[start:end] = pbuf + 1

end=len(buf)
half = int(len(buf)/2)

start_time = time.time()
t1 = threading.Thread(target=buf_plus, args=(buf[:half], 0, half))
t2 = threading.Thread(target=buf_plus, args=(buf[half:],  half, end))
t1.start(); t2.start()
t1.join(); t2.join()
end_time = time.time()
print (buf.shape)
print("buf+=1: %f seconds (2 threads)" % 
      (end_time - start_time)) 

(500000000,)
buf+=1: 11.318131 seconds (2 threads)


In [28]:
start_time = time.time()
ctx = mp.get_context('fork')
q = ctx.Queue()
p1 = ctx.Process(target=buf_plus, args=(buf[:half], 0, half))
p2 = ctx.Process(target=buf_plus, args=(buf[half:],  half, end))
p1.start(); p2.start()
p1.join(); p2.join()
end_time = time.time()
print("buf+=1: %f seconds (2 processes)" % 
      (end_time - start_time)) 

buf+=1: 14.597568 seconds (2 processes)
